> Misal lagi training LLM yg gede 7B params, utk stability(trutama pas loss landscape yg noisy), kita needs effective batch size yg gede (let say 512 samples per step) -> tapi `GPU cuma muat 8 samples sblm VRAM jeblug (oom)`. Nah kenapa VRAM jeblug? ya iyalah pas forward pass dia kan nyimpen activation func dri stiap layer (buat dipake pas backprop). More `big si batch size kan means bkl gede juga act function yg disimpen`. Jadi problemnya itu:<br>
- kita needs batch size yg gede biar stabil nggak noisy on one hand kita juga punya kterbatasan memori dan normalnya jalan di BS kecil<br>
so disini si `Grad Accumulation` datang sbg hero..

# Intuisi

- Analogi: kita mw ngitung rata2 ujian dari 1000 mhsiswa, tapi meja kita cuma muat 10 kertas skaligus. Solusinya gimana? yaa kita cek 10 kertas, catet total nilainya (kaga kita bagi lsg dg 10) itunggg aja terus sampe habis 1000 msiswa it, baru diakhir dibagi dengan 1000
- Analogi di graident, jadi setiap `minibatch gradient diitung dan diaccumulate` (ditambah) ke buffer `.grad` without manggil si `optimizer.step()` dulu. Nah setelah selesai `N minibatch` baru `optimizer.step()` dipanggil skali, ya mo sgimanapun gede batchnya no prob
- Keynya adalah `loss.backward()` itu ngeaccumulate ke `.grad` bukan overwrite